## Spark's Query Plan

<img src="https://raw.githubusercontent.com/afaqueahmad7117/spark-experiments/7ec30cf18fe06373cb5c2cf69db90ece04b08d51/data/spark-execution.png">

1. **SQL / DataFrame** — code enters as either SQL (parsed to AST) or DataFrame API calls
2. **Unresolved Logical Plan** — syntax checked, but table/column names not yet validated
3. **Catalog check** — Spark verifies tables, columns, and data types actually exist
4. **Logical Plan** — fully resolved plan once validation passes
5. **Optimized Logical Plan** — Catalyst applies rule-based optimizations (projection pushdown, filter pushdown, etc.)
    - Projection pushdown: if you wrote SELECT * but only use 5 columns downstream, Spark rewrites the plan to only read those 5.
    - Predicate/filter pushdown: if you filter partway through your chain, Spark tries to push that filter as close to the data source as possible, so fewer rows get pulled in the first place.
6. **Physical Plans** — multiple candidate execution strategies generated (e.g., different join types)

    Note: Executor count (compute resources, set at cluster config) and shuffle partition count (spark.sql.shuffle.partitions, decided at physical plan/runtime) are independent — task scheduler assigns partition-tasks to free executor cores dynamically, no partition is "owned" by an executor.
7. **Cost Model → Selected Physical Plan** — best plan chosen based on cost estimates (data size, stats)
8. **Code Generation → RDDs** — selected plan compiled into executable code, runs on cluster as RDD operations


**Note:** Executor count vs. shuffle partition count

- **Executor count** — compute resources, set at cluster config (before query runs)
- **Shuffle partition count** — `spark.sql.shuffle.partitions`, decided at physical plan / runtime
- These two are **independent** of each other
- The **task scheduler** dynamically assigns partition-tasks to free executor cores
- No partition is "owned" by a fixed executor


In [ ]:
from pyspark.sql import functions as F

transaction_path = "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/pyspark_optimization/data/data_skew/transactions.parquet/"
customer_path =  "abfss://testcontainer@storageinswa01us2dev.dfs.core.windows.net/pyspark_optimization/data/data_skew/customers.parquet/"

df_transactions = spark.read.format('parquet').load(transaction_path)
df_customers = spark.read.format('parquet').load(customer_path)

## Narrow and Wide transformations

<img src="https://www.sparkplayground.com/images/narrow-wide-transformations.png" height=600/>

In [ ]:
df_narrow_transform = (
    df_customers
    .filter(F.col("city") == "boston")
    .withColumn("first_name", F.split("name", " ").getItem(0))
    .withColumn("last_name", F.split("name", " ").getItem(1))
    .withColumn("age", F.col("age") + F.lit(5))
    .select("cust_id", "first_name", "last_name", "age", "gender", "birthday")
)

df_narrow_transform.show(5, False)
df_narrow_transform.explain(True)

**Read direction: Bottom to top**

Execution order:
1. `FileScan parquet` (bottom) — first
2. `ColumnarToRow`
3. `Filter`
4. `Project` (top) — last

---

**High-level explanation, step by step**

| Step | What it does |
|---|---|
| **FileScan parquet** | Reads the Parquet file. `PushedFilters: [IsNotNull(city), EqualTo(city,boston)]` — filters pushed down to the source, so fewer rows get pulled off disk in the first place. |
| **ColumnarToRow** | Parquet stores data column-wise; Spark converts to row format because downstream transformations (filter, split, arithmetic) are easier to apply row-by-row. |
| **Filter** | `isnotnull(city) AND city = boston` — the redundant safety-net filter. Even though the filter was pushed to the source, Spark re-applies it here as a correctness guarantee (not all data sources honor pushed filters reliably). |
| **Project** | Builds the final output columns: splits `name` into `first_name`/`last_name`, casts `age` to double and adds 5, passes through `gender`, `birthday`. This is where `withColumn` calls actually get computed. |

**No `Exchange` step** — confirms a pure narrow transformation pipeline (filter + column derivation only): no shuffle, everything computed independently per partition, fused into one physical stage (`*(1)`).


## Wide Transformations
1. Repartition
2. Coalesce
3. Joins
4. GroupBy
    - count
    - countDistinct
    - sum

In [ ]:
df_transactions.rdd.getNumPartitions()

In [ ]:
df_transactions.repartition(24).explain(True)

## Physical Plan Summary — Repartition (Wide Transformation)

**Code:** `df_transactions.repartition(24)`

**Plan:**

AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(24), REPARTITION_BY_NUM
+- FileScan parquet [...]


**Bottom to top:**

| Step | What happens |
|---|---|
| **FileScan parquet** | Reads file, arrives as **12 partitions** (from `spark.sql.files.maxPartitionBytes` ≈ 128MB/chunk — a default Spark config, not set by us). No filters applied (`DataFilters: []`). |
| **Exchange – RoundRobinPartitioning(24)** | The shuffle. Rows from all 12 input partitions are redistributed into **24 new partitions**. |
| **AdaptiveSparkPlan isFinalPlan=false** | AQE may revise this plan at runtime based on actual execution stats. |

---

### Round robin partitioning — how it actually works

- All rows across the **12 input partitions are pooled into one logical stream** (not reshuffled partition-by-partition independently)
- Rows are then **dealt out sequentially, one at a time, in round-robin fashion** into the 24 output partitions: row 1 → partition 0, row 2 → partition 1, ... row 25 → partition 0 again, and so on
- **No key is used** — unlike hash partitioning (joins/groupBy), round-robin ignores column values entirely; it only cares about even **row-count distribution**
- Because rows are pooled across *all* input partitions before redistribution, a row from old partition 3 can land in a new partition that lives on a completely different executor — this cross-executor data movement is exactly why it needs a real **shuffle** (network + disk I/O), not just local reassignment
- Goal: **balanced partition sizes**, purely by count — useful when repartitioning for parallelism, not for co-locating related keys (that's what hash partitioning is for)

**No filters, no keys involved** — confirms this plan is a pure repartition operation, nothing else.




<img src="https://miro.medium.com/v2/resize:fit:750/format:webp/1*Dw2zXANXra43BOu0l9ToYQ.gif"/>


## Coalesce

**Why coalesce over repartition:** Avoids shuffle (a costly operation).

**How:** Merges partitions *within the same executor* — reducing total partition count without moving data across the network.

---

**Example 1 — no shuffle (mild reduction)**

- 3 executors, 12 partitions total (4 per executor)
- `coalesce(3)` → target (3) = number of executors (3)
- Each executor merges its own 4 partitions locally → **1 partition per executor → 3 total partitions**
- No shuffle needed — every executor produces its output using only its own data

---

**Example 2 — shuffle involved (aggressive reduction)**

- 4 executors, `coalesce(2)` → target (2) < number of executors (4)
- At least 2 executors **cannot** produce their own output partition locally
- Spark picks 2 "surviving" executors to host the final partitions
- The other 2 executors' data must be **shuffled (moved over the network)** into those surviving executors

**Key takeaway:** In aggressive partition reduction, coalesce is forced to involve shuffle — it can no longer stay purely local-merge.


In [ ]:
df_transactions.coalesce(1).explain(True)

## Joins

To join two tables on a key, matching rows must end up being **compared to each other** — and Spark can only compare rows that live **within the same partition** (no cross-partition, row-by-row comparison exists).

So the fundamental goal of every join strategy is the same:

> **Get rows with the same key value physically co-located in the same partition, one way or another.**

Only *how* that co-location is achieved differs:

- **Sort-Merge Join** — both tables are shuffled using hash partitioning on the key, so matching keys land in the same numbered partition across both tables
- **Broadcast Hash Join** — instead of moving the large table, the *entire small table* is copied to every partition of the large table, so every partition already has full access to all possible matching keys, with no shuffle needed for the large side

Different mechanism, same underlying necessity: **before a join can happen locally within a partition, every row that might match must somehow be present in that same partition.**


In [ ]:
df_sales = (
    df_transactions.alias("T1")
    .join(
        df_customers.alias("T2"),
        on = F.col("T1.cust_id") == F.col("T2.cust_id"),
        how = 'inner'
    )
)
    
df_sales.explain(True)

## Group By

**Example:** `df.groupBy("city").count()`

---

### 1. Local partial aggregation (per partition, no shuffle)

- Each input partition independently computes a **partial result** using only the data it already has
- Pure in-memory computation — no network movement, no shuffle
- Example (Mumbai, 3 of many partitions):
  - P1: Mumbai → 10
  - P2: Mumbai → 30
  - P3: Mumbai → 40

**Physical plan step:** `HashAggregate(keys=[city], functions=[partial_count(1)])`

---

### 2. Shuffle — co-locate same keys (Exchange)

- Goal: get every partial result for the **same key** into **one** destination partition, so they can be combined
- Destination partition decided by `hash(key) % shuffle_partitions` (default 200) — e.g. `hash("Mumbai") % 200` → some output partition, unrelated to which input partition the data came from
- **What actually crosses the network here is small partial values (10, 30, 40) — NOT the original raw rows**
- This is the core optimization: shuffle payload shrinks from millions of raw rows to one small number per partition per key

**Physical plan step:** `Exchange hashpartitioning(city, 200)`

---

### 3. Final aggregation (per destination partition, no shuffle)

- Each destination partition now holds all the partial values for its assigned keys
- Sums (or combines) them locally, in-memory, to produce the true final result
- Example: Mumbai → 10 + 30 + 40 = **70**

**Physical plan step:** `HashAggregate(keys=[city], functions=[count(1)])`

---

### Why this works — the underlying requirement

- `count`, `sum`, `min`, `max` are **associative/commutative** — safe to compute in partial pieces on different partitions, then combine later
- Operations that aren't (e.g. `median`) can't be pre-aggregated this way — they're forced to shuffle raw values, which is far more expensive

---

### Net effect

| | Data moved during shuffle |
|---|---|
| Without partial aggregation | Every raw row |
| With partial aggregation | One small partial value per partition per key |

**Full pattern in the physical plan:**
```
HashAggregate(final)
+- Exchange hashpartitioning(key, N)
   +- HashAggregate(partial)
      +- FileScan
```


In [ ]:
df_city_counts = df_transactions.groupBy("city").count()
df_city_counts.explain(True)

In [ ]:
df_city_counts.show(5)

## Pushdown Filter/Projection Failures

**Case 1 — Nested struct/map access**
`df.filter(city['value'] == 'mumbai')`
- Parquet can't interpret key-based access inside a struct/map at the source level
- Must materialize full struct in Spark first, then filter → **no pushdown**

**Case 2 — Cast before filter**
`df.filter(cast(age as int) < 34)`
- Source stores `age` as string; filter compares as int → type mismatch
- Cast must run in Spark after reading → only `IsNotNull` pushes, not the actual comparison

**Fix — correct schema at write time (Silver layer)**
- Store `age` as proper int type → no cast needed → full filter pushes
- Flatten nested structs into separate columns (`city_type`, `city_value`) → plain column filters push

**Why it matters:** fixing schema once at ingestion means every downstream query gets pushdown automatically — no need to rely on careful filter syntax later.